Null Hypothesis: The playcount per listener ratio is the same for artists who use AI and artists that dont use AI.

Alternative Hypothesis: Artists that use AI have lower playcount per listener ratio.

In [19]:
!pip install matplotlib==3.10.6 \
numpy==2.3.3\
pandas==2.3.3



---



In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = Path().resolve()
ENRICHED_PATH = DATA_DIR / "deezer_lastfm_enriched.csv"

if not ENRICHED_PATH.exists():
    raise FileNotFoundError(f"Could not find dataset at {ENRICHED_PATH}")

plt.style.use("seaborn-v0_8-whitegrid")
pd.options.display.float_format = "{:.2f}".format



---



In [2]:
preview_lines = []
with ENRICHED_PATH.open() as f:
    for line in f:
        preview_lines.append(line.strip())
        if len(preview_lines) == 6:
            break

for line in preview_lines:
    print(line)


artist_id,artist_name,artist_nb_albums,artist_fans_deezer,album_id,album_title,album_url,release_date,nb_tracks,genre,deezer_fans,ai_flagged,ai_flag_text,scrape_error,search_query,lastfm_artist_found,lastfm_artist_name,artist_playcount,artist_listeners,artist_url,album_playcount,album_match_type,album_match_score
74069182,Sombre Ensemble,5,22,174228742,"Fête d'Halloween 2020: Ambiance effrayante avec bruits de fantômes, âmes damnées, clowns killer",https://www.deezer.com/en/album/174228742,2020-10-02,15,,52,False,,,ambient 2025,True,Sombre Ensemble,194,127,https://www.last.fm/music/Sombre+Ensemble,3,exact,100
103467092,Łil Cødeìnê,1,0,751670411,Soul (Live 2025),https://www.deezer.com/en/album/751670411,2025-04-26,1,Rap/Hip Hop,0,False,,,soul 2025,False,,,,,,no_tracks_found,0
1290704,Capoeira Experience,13,937,562325811,Capoeira Eletrônica 2024,https://www.deezer.com/en/album/562325811,2024-03-22,18,,19,False,,,electronic 2025,True,Capoeira Experience,22642,3090,https://www.last.fm/musi



---



In [8]:
mydata = pd.read_csv(ENRICHED_PATH)
mydata.head()
mydata.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 23 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   artist_id            500 non-null    int64  
 1   artist_name          500 non-null    object 
 2   artist_nb_albums     500 non-null    int64  
 3   artist_fans_deezer   500 non-null    int64  
 4   album_id             500 non-null    int64  
 5   album_title          500 non-null    object 
 6   album_url            500 non-null    object 
 7   release_date         500 non-null    object 
 8   nb_tracks            500 non-null    int64  
 9   genre                432 non-null    object 
 10  deezer_fans          500 non-null    int64  
 11  ai_flagged           500 non-null    bool   
 12  ai_flag_text         86 non-null     object 
 13  scrape_error         1 non-null      object 
 14  search_query         500 non-null    object 
 15  lastfm_artist_found  500 non-null    boo



---



In [20]:
columns_of_interest = ["artist_name", "artist_fans_deezer", "lastfm_artist_name", "artist_listeners", "artist_playcount"]
summary = mydata[columns_of_interest].describe(include="all")
summary

,artist_name,artist_fans_deezer,lastfm_artist_name,artist_listeners,artist_playcount
count,500,500.00,453,453.00,453.00
unique,499,NaN,452,NaN,NaN
top,Çeşitli Sanatçılar,NaN,Çeşitli Sanatçılar,NaN,NaN
freq,2,NaN,2,NaN,NaN
mean,NaN,72072.55,NaN,103737.34,6701488.98
std,NaN,679003.03,NaN,526923.96,53810546.16
min,NaN,0.00,NaN,1.00,1.00
25%,NaN,1.00,NaN,24.00,101.00
50%,NaN,24.50,NaN,506.00,2974.00
75%,NaN,766.50,NaN,12058.00,127316.00




---



In [10]:
top_ai_artists = (
    mydata
    .query("ai_flagged == True and lastfm_artist_found == True")
    .sort_values(by="artist_listeners", ascending=False)
    .loc[:, ["artist_name", "album_title", "artist_fans_deezer", "artist_listeners", "artist_playcount"]]
    .head(5)
)
top_ai_artists

,artist_name,album_title,artist_fans_deezer,artist_listeners,artist_playcount
366,Ray,Dance Pop 2025,2361,119125.00,3157900.00
433,Happy Birthday Songs,Happy Birthday (Disco Pop Version 2025),399,46359.00,126968.00
340,Sissy,In My Soul (Remastered 2025),202,16125.00,201159.00
406,World Music,Türkçe Pop 2025,14509,9681.00,87485.00
44,Old Soul,Silent Night EDM 2025,233,7741.00,66099.00




---



In [13]:
top_non_ai_artists = (
    mydata
    .query("ai_flagged == False and lastfm_artist_found == True")
    .sort_values(by="artist_listeners", ascending=False)
    .loc[:, ["artist_name", "album_title", "artist_fans_deezer", "artist_listeners", "artist_playcount"]]
    .head(5)
)
top_non_ai_artists

,artist_name,album_title,artist_fans_deezer,artist_listeners,artist_playcount
159,Linkin Park,From Zero (Deluxe Edition),12045663,6914928.00,740469082.00
258,Pink Floyd,The Dark Side Of The Moon (Live at Wembley 197...,6736100,5541702.00,608171106.00
435,Prince,Pop Life (Extended Version) (2025 Remaster),1325937,3339586.00,103213921.00
142,Ray Charles,Ingredients In A Recipe For Soul (2025 Remaster),2913463,2834663.00,44644669.00
238,Hans Zimmer,Blade Runner 2049 (Original Motion Picture Sou...,2117077,2712086.00,144763149.00




---



In [14]:
ai_artist_avg_listeners= mydata.groupby("ai_flagged")["artist_listeners"].mean()
ai_artist_avg_listeners

,artist_listeners
ai_flagged,
False,121192.77
True,3173.25




---



In [16]:
listeners_summary_by_ai = mydata.groupby("ai_flagged")["artist_listeners"].agg(["mean", "median", "count"])
listeners_summary_by_ai

,mean,median,count
ai_flagged,,,
False,121192.77,1003.50,386
True,3173.25,20.00,67




---



In [18]:
playcount_summary_by_ai = mydata.groupby("ai_flagged")["artist_playcount"].agg(["mean", "median", "count"])
playcount_summary_by_ai

,mean,median,count
ai_flagged,,,
False,7855062.86,6781.00,386
True,55526.03,54.00,67


   

---








In [21]:
"""calculated the plays-per-listener ratio and added it as a new column to my dataset"""
mydata_with_ratio = mydata.assign(playcount_to_listener_ratio = mydata["artist_playcount"] / mydata["artist_listeners"])







---









In [23]:
ratio_summary_by_ai = mydata_with_ratio.groupby("ai_flagged")["playcount_to_listener_ratio"].agg(["mean", "median", "count"])
ratio_summary_by_ai

,mean,median,count
ai_flagged,,,
False,17.05,6.47,386
True,6.43,3.25,67




---



In [27]:
mydata_with_ratio_for_albums = mydata.assign(
    album_playcount_to_listener_ratio = mydata["album_playcount"] / mydata["artist_listeners"]
)

album_ratio_summary_by_ai = mydata_with_ratio_for_albums.groupby("ai_flagged")["album_playcount_to_listener_ratio"].agg(["mean", "median", "count"])
album_ratio_summary_by_ai

,mean,median,count
ai_flagged,,,
False,2.32,0.06,386
True,0.91,0.00,67
